##### Path

In [27]:
result_path = joinpath(@__DIR__, "..", "..", "Result", "Unfitted_FEM","2D_Plate_With_Hole_CutFEM")
isdir(result_path) || mkpath(result_path)

true

##### Packages

In [28]:
using Gridap
using GridapEmbedded
using GridapTopOpt
using LinearAlgebra

Gridap.TensorValues.SymFourthOrderTensorValue

SymFourthOrderTensorValue

In [29]:
# doughnut(R,r;x0=zero(Point{3,typeof(R)}),name="doughnut")

# popcorn(r0=0.6, σ=0.2, A=2, x0=zero(Point{3,typeof(r0)}), name="popcorn")

# sphere(R;x0=zero(Point{3,eltype(R)}),name="sphere")

# disk(R;x0=zero(Point{2,eltype(R)}),name="disk")

# cylinder(R;x0=zero(Point{3,eltype(R)}),v=VectorValue(1,0,0),name="cylinder")

# plane(x0=Point(0,0,0),v=VectorValue(1,0,0),name="plane")

# square(L=1,x0=Point(0,0),name="square",edges=["edge_i" for i in 1:4])

# quadrilateral(x0=Point(0,0),d1=VectorValue(1,0),d2=VectorValue(0,1),name="quadrilateral")

# cube(L=1,x0=Point(0,0,0),name="cube")

# tube(R,L;x0=zero(Point{3,typeof(R)}),v=VectorValue(1,0,0),name="tube")

##### Background Mesh

In [30]:
L = 1000.0
B = 400.0

h = 25                      # Mesh size parameters
nx = floor(L/h)
ny = floor(B/h)

domain = (-L/2, L/2, -B/2, B/2)
partition = (nx, ny)
bgmodel = CartesianDiscreteModel(domain,partition)
f_Γ_D(x) = x[1] == -L/2
f_Γ_N(x) = x[1] == L/2 
update_labels!(1,bgmodel,f_Γ_D,"Gamma_f_D")
update_labels!(2,bgmodel,f_Γ_N,"Gamma_f_N")

In [31]:
check_path = joinpath(result_path, "Model_Check")
isdir(check_path) || mkpath(check_path)
writevtk(bgmodel, joinpath(check_path, "Model_Check")) 

3-element Vector{Vector{String}}:
 ["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\2D_Plate_With_Hole_CutFEM\\Model_Check\\Model_Check_0.vtu"]
 ["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\2D_Plate_With_Hole_CutFEM\\Model_Check\\Model_Check_1.vtu"]
 ["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\2D_Plate_With_Hole_CutFEM\\Model_Check\\Model_Check_2.vtu"]

##### Implicit Boundary

In [32]:
# # IMPLICIT RECTANGLE IN 2D
# ##########################
# # L for x axis
# # B for y axis

# function Rectangle2D(;L=1, B=1, x0=Point(0.0, 0.0), name="Rectangle2D")

#   e1 = VectorValue(1,0)
#   e2 = VectorValue(0,1)

#   line1 = plane(x0 = x0 - 0.5*B*e2, v = -e2, name="bottom")
#   line2 = plane(x0 = x0 + 0.5*B*e2, v = +e2, name="top")
#   line3 = plane(x0 = x0 - 0.5*L*e1, v = -e1, name="left")
#   line4 = plane(x0 = x0 + 0.5*L*e1, v = +e1, name="right")

#   geo12 = intersect(line1,line2)
#   geo34 = intersect(line3,line4)

#   intersect(geo12,geo34,name=name)

# end

In [33]:
R = 75                # Radius
L = 1000.0            # Length
B = 400.0             # Breadth

const C0 = Point(0.0,0.0)
const p1 = Point(-L/2,-B/2)
const e1 = VectorValue(L,0.0)
const e2 = VectorValue(0.0,B)

geo1 = disk(R,x0=C0)

geo2 = quadrilateral(
    x0 = p1,
    d1 = e1,
    d2 = e2
)

Ω = setdiff(geo2,geo1)

AnalyticalGeometry(Node((:-, "", nothing),Node((:∩, "quadrilateral", nothing),Node((:∩, "", nothing),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{2, Float64}, VectorValue{2, Float64}}((-500.0, -200.0), (0.0, -1.0)), "edge1", nothing)),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{2, Float64}, VectorValue{2, Float64}}((-500.0, 200.0), (-0.0, 1.0)), "edge2", nothing))),Node((:∩, "", nothing),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{2, Float64}, VectorValue{2, Float64}}((-500.0, -200.0), (-1.0, 0.0)), "edge3", nothing)),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{2, Float64}, VectorValue{2, Float64}}((500.0, -200.0), (1.0, -0.0)), "edge4", nothing)))),Leaf((GridapEmbedded.LevelSetCutters.var"#diskfun#10"{VectorValue{2, Float64}, Int64}((0.0, 0.0), 75), "disk", GridapEmbedded.LevelSetCutters.BoundingBox{2, Float64}((-75.75, -75.75), (75.75, 75.75))))))

##### Cut Geometry

In [34]:
# Background Model , Analytical Geometry
cutgeo = cut(bgmodel,Ω)

EmbeddedDiscretization()

In [35]:
Γ       = EmbeddedBoundary(cutgeo)
Ω_act   = Triangulation(cutgeo,ACTIVE)
Ω_bg    = Triangulation(bgmodel)
Ω       = Triangulation(cutgeo,PHYSICAL)

AppendedTriangulation()

In [36]:
writevtk(Γ,     joinpath(result_path,"Γ"))
writevtk(Ω_act, joinpath(result_path,"ACTIVE_Triangulation"))
writevtk(Ω_bg,  joinpath(result_path,"Background_Triangulation"))
writevtk(Ω,     joinpath(result_path,"Physical_Triangulation"))

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\2D_Plate_With_Hole_CutFEM\\Physical_Triangulation.vtu"],)

##### FE Formulation

In [37]:
order = 1
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
Vstd = TestFESpace(Ω_act,reffe,conformity=:H1)

UnconstrainedFESpace()

In [38]:
# Aggregate the cut cells
strategy = AggregateAllCutCells()
aggregates = aggregate(strategy,cutgeo);

In [39]:
colors = color_aggregates(aggregates,bgmodel)
Ω_bg = Triangulation(bgmodel)  
writevtk(Ω_bg,joinpath(result_path,"aggs_on_bg_trian"),celldata=["aggregate"=>aggregates,"color"=>colors])

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\2D_Plate_With_Hole_CutFEM\\aggs_on_bg_trian.vtu"],)

In [40]:
V = AgFEMSpace(Vstd,aggregates)
U = TrialFESpace(V)

FESpaceWithLinearConstraints()

In [41]:
degree = 2*order
dΩ = Measure(Ω,degree)
n_Γ = get_normal_vector(Γ)
dΓ = Measure(Γ,degree)

# Neumann
Γ_N = BoundaryTriangulation(bgmodel,tags="Gamma_f_N")
dΓ_N = Measure(Γ_N,2)
# Drichlet
Γ_D = BoundaryTriangulation(bgmodel,tags="Gamma_f_D")
dΓ_D = Measure(Γ_D,2)
n_Γ_D = get_normal_vector(Γ_D)

GenericCellField():
 num_cells: 16
 DomainStyle: ReferenceDomain()
 Triangulation: BoundaryTriangulation()
 Triangulation id: 16192058765957043888

In [42]:
h = get_element_diameter_field(bgmodel)
γg = 0.1
γD = 10.0

10.0

In [43]:
function  ElasFourthOrderConstTensor(E,ν,PlanarState)# 1 for  Plane  Stress  and 2 Plane  Strain  Condition
    if  PlanarState  == 1
        C1111 =E/(1-ν*ν)
        C1122 = (ν*E)/(1-ν*ν)
        C1112 = 0.0
        C2222 =E/(1-ν*ν)
        C2212 = 0.0
        C1212 =E/(2*(1+ν))
    elseif  PlanarState  == 2
        C1111 = (E*(1-ν*ν))/((1+ν)*(1-ν-2*ν*ν))
        C1122 = (ν*E)/(1-ν-2*ν*ν)
        C1112 = 0.0
        C2222 = (E*(1-ν))/(1-ν-2*ν*ν)
        C2212 = 0.0
        C1212 =E/(2*(1+ν))
    end
    C_ten = SymFourthOrderTensorValue(C1111 ,C1112 ,C1122 ,C1112 ,
        C1212 ,C2212 ,C1122 ,C2212 ,C2222)
    return   C_ten
end

ElasFourthOrderConstTensor (generic function with 1 method)

In [44]:
E = 21000.0
ν = 0.3
const  C = ElasFourthOrderConstTensor(E,ν,2)

λ = (E*ν)/((1+ν)*(1-2ν))
μ = E/(2*(1+ν))

g = VectorValue(10,0)                       # Force (Traction)

VectorValue{2, Int64}(10, 0)

In [45]:
a(u,v) =
  ∫( ε(v) ⊙ C ⊙ ε(u) )dΩ +
  ∫( γg * h * (λ+2μ) *
     ( jump(n_Γ⋅∇(u)) ⋅ jump(n_Γ⋅∇(v)) )
   )dΓ -
  ∫( ((C ⊙ ε(u)) ⋅ n_Γ_D) ⋅ v )dΓ_D -
  ∫( ((C ⊙ ε(v)) ⋅ n_Γ_D) ⋅ u )dΓ_D +
  ∫( γD * (λ+2μ) / h * (u⋅v) )dΓ_D
l(v) = ∫( v⋅g )dΓ_N

l (generic function with 1 method)

In [46]:
op = AffineFEOperator(a, l, U, V)
uh = solve(op)

SingleFieldFEFunction():
 num_cells: 624
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 3652506022664196768

In [47]:
writevtk(Ω,joinpath(result_path,"results.vtu"),
    cellfields=[
        "Displacement"=>uh,
        "σ"=>(C ⊙ ε(uh))]
        )  

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\..\\..\\Result\\Unfitted_FEM\\2D_Plate_With_Hole_CutFEM\\results.vtu"],)